In [1]:
import sqlite3
import os

# 数据库路径
db_path = "/Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/cmidata.db"

# 检查文件是否存在
if not os.path.exists(db_path):
    print(f"❌ 文件不存在: {db_path}")
else:
    print(f"✅ 找到数据库: {db_path}")

# 建立连接
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 获取所有表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\n📋 共 {len(tables)} 张表：")
for t in tables:
    print(f"   - {t[0]}")

✅ 找到数据库: /Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/cmidata.db

📋 共 1 张表：
   - purchase_orders


In [5]:
import pandas as pd

# 显示每张表的字段定义
for (table_name,) in tables:
    print(f"\n{'='*60}")
    print(f"📊 表: {table_name}")
    print(f"{'='*60}")

    # PRAGMA table_info 返回：列序号, 列名, 数据类型, 是否非空, 默认值, 是否主键
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()

    # 用 DataFrame 展示，更清晰
    df_cols = pd.DataFrame(columns, columns=["序号", "列名", "类型", "非空", "默认值", "主键"])
    df_cols["主键"] = df_cols["主键"].map({1: "✅", 0: ""})
    df_cols["非空"] = df_cols["非空"].map({1: "✅", 0: ""})
    display(df_cols[["列名", "类型", "非空", "主键"]])

    # 同时显示行数
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"   行数: {count}")



📊 表: purchase_orders


,列名,类型,非空,主键
0,采购订单编号,TEXT,,
1,IBOSS产品类型,TEXT,,
2,订单状态,TEXT,,
3,承办人,TEXT,,
4,承办部门,TEXT,,
...,...,...,...,...
80,是否涉及特殊审批,REAL,,
81,特殊审批单号,REAL,,
82,实际取消/终止日期,REAL,,
83,是否有付款需求,REAL,,


   行数: 802


In [5]:
# 先找出包含"供应"的字段名
cursor.execute(f"PRAGMA table_info(purchase_orders)")
all_cols = [row[1] for row in cursor.fetchall()]
supplier_cols = [c for c in all_cols if "供应" in c]
print("🔍 包含'供应'的字段：", supplier_cols)

# 优先用"供应商名称"字段统计
target_col = "供应商名称" if "供应商名称" in supplier_cols else supplier_cols[0]

df_suppliers = pd.read_sql_query(
    f"SELECT DISTINCT `{target_col}` FROM purchase_orders WHERE `{target_col}` IS NOT NULL AND `{target_col}` != ''",
    conn
)
print(f"\n📊 供应商数量（按「{target_col}」去重）: {len(df_suppliers)}")
display(df_suppliers)

🔍 包含'供应'的字段： ['供应商审批单批号', '供应商名称', '供应商电路编号', '供应商电路编号(行)', '供应商编号', '是否客户指定供应商', '客户供应商是否关联公司', '供应商退租邮箱', '供应商取消/终止日期', '供应商罚金']

📊 供应商数量（按「供应商名称」去重）: 59


,供应商名称
0,"GUOLI Holdings Co., Limited"
1,Telekomunikasi Indonesia International Pte. Ltd.
2,ソフトバンク株式会社
3,KDDI Corporation
4,雲鐳 (香港)有限公司
5,Cypress Telecom Limited
6,曉通網路科技（香港）有限公司
7,世纪二千网络科技有限公司
8,SICITEL LIMITED
9,ATTO DIGITAL INFORMATION TECHNOLOGY (HONG KONG...


In [4]:
# 查找跟金额/收入相关的字段
cursor.execute("PRAGMA table_info(purchase_orders)")
all_cols = [row[1] for row in cursor.fetchall()]
money_cols = [c for c in all_cols if any(k in c for k in ["金额", "收入", "费用", "价格", "价", "合计", "总", "amount", "cost", "revenue", "price", "金额"])]
print("💰 包含金额/费用相关的字段：")
for c in money_cols:
    print(f"   - {c}")

💰 包含金额/费用相关的字段：
   - 报价单号
   - 采购订单总金额（港币）
   - NRC里程碑金额(原币不含税)
   - 突发流量价格


In [10]:
# ⚠️ 同一订单编号在表中有多行明细，必须先按订单去重，否则金额会被重复计算
df_orders = pd.read_sql_query("""
    SELECT DISTINCT
        `采购订单编号`,
        `供应商名称`,
        `采购订单总金额（港币）` AS 订单金额,
        `Service Type`
    FROM purchase_orders
    WHERE `供应商名称` IS NOT NULL AND `供应商名称` != ''
""", conn)

print(f"📋 去重后订单数: {len(df_orders)}（原表有大量重复行）")

# 用 Pandas 聚合：按供应商统计金额 + 收集服务类型列表
df_revenue = df_orders.groupby("供应商名称", as_index=False).agg(
    订单数=("采购订单编号", "nunique"),
    总金额_HKD=("订单金额", "sum"),
    服务类型列表=("Service Type", lambda x: sorted(set(s for s in x if s and str(s).strip())))
)

# 服务类型格式化为逗号分隔的字符串，空值显示为 "未标注"
df_revenue["服务类型"] = df_revenue["服务类型列表"].apply(
    lambda lst: ", ".join(lst) if lst else "未标注"
)

# 排序
df_revenue = df_revenue.sort_values("总金额_HKD", ascending=False).reset_index(drop=True)

# 格式化金额显示
total_hkd = df_revenue["总金额_HKD"].sum()
df_revenue["总金额_HKD_display"] = df_revenue["总金额_HKD"].round(2).apply(lambda x: f"{x:,.2f}")

print(f"\n📊 供应商收入统计（共 {len(df_revenue)} 家供应商）")
print(f"   总计订单数: {df_revenue['订单数'].sum()}")
print(f"   总金额 (HKD): {total_hkd:,.2f}")

# 显示结果（含服务类型列）
display(df_revenue[["供应商名称", "订单数", "总金额_HKD_display", "服务类型"]].rename(
    columns={"总金额_HKD_display": "总金额 (HKD)"}
))

📋 去重后订单数: 251（原表有大量重复行）

📊 供应商收入统计（共 59 家供应商）
   总计订单数: 234
   总金额 (HKD): 37,084,467.22


,供应商名称,订单数,总金额 (HKD),服务类型
0,Smart Vic Limited,43,"5,912,123.31","其他, 现场集成服务, 集成实施服务"
1,曉通網路科技（香港）有限公司,25,"2,653,217.63","AAS CPE, AAS Other, CPE, 现场集成服务, 集成实施服务"
2,LINKCIRCLE HONGKONG COMPANY LIMITED,6,"2,448,217.92","Local Loop, Other"
3,METRO INTEGRATED TECH INC.,3,"1,826,037.95","ICT终端及设备, 现场集成服务, 集成实施服务"
4,Beijing Sinonet Information Technology Limited,1,"1,676,303.35",Port Sec
5,Digital Plan System,1,"1,617,464.58","ICT终端及设备, 集成实施服务"
6,"GUOLI Holdings Co., Limited",9,"1,591,610.60","CPE, Port Sec"
7,SICITEL LIMITED,1,"1,488,466.44",Port Sec
8,台灣固網股份有限公司,3,"1,398,263.34","Local Loop, Port Sec-IP"
9,FPT Telecom,1,"1,199,828.28",Local Loop
